# ShowPtycho → independent iterative regions

**Local workflow template:** run this notebook with your acquisition and GPU. The documentation site shows its source; it cannot execute a private acquisition or the separately installed MyPtycho package.

Load a 4D-STEM acquisition, review it in ShowPtycho, then initialize one myptycho run per fitted region. Install both `quantem.widget` and `myptycho` in the same notebook environment. GPU selection is automatic: CUDA or Apple MPS.

Change `scan.qem` to your file. An ARINA H5 master uses the same `io.load("scan_master.h5")` call; keep its data shards beside it. For a generic H5 dataset use `io.load("scan.h5", dataset_path="/entry/data", scan_shape=(512, 512))` with your actual dataset and scan dimensions. The loaded object's table reports file calibration and storage before SSB starts.

A QEM with saved voltage, convergence angle and scan step reuses them. For H5 or QEM with missing values, supply the measured values to `DirectPtycho`, for example:

```python
ptycho = DirectPtycho(data, voltage_kV=200, semiangle_mrad=30,
          scan_sampling_A=(0.25, 0.25), scan_detector_rotation_deg=17)
```

These numbers are examples, not defaults for your microscope. Detector sampling is reused when saved and fitted from the probe disk when missing; do not type an estimated detector step. The launch table names file values, values used and override sources. A supplied rotation is a reviewed calibration, not automatically proven by loading the file.

In ShowPtycho, fit the overview jointly, enable **Local Tilt**, draw or resize regions, and wait for their joint fits to finish. Each regional fit uses one joint fit with seed 42, 200 trials and quantem.gpu's exact-gradient refinement. Press **Save** to restore this reviewed state after a restart. A new session needs this interactive step before running the handoff cell; a matching saved session restores its fits.

The example below explicitly assumes a **25 nm, 20-slice, single-coherent-probe model** and runs eight epochs to check the workflow. Choose the specimen model for your experiment and increase epochs after reviewing convergence. Orthogonal multi-mode probe learning is not yet supported on MPS; this example uses one mode on both platforms, without a CPU fallback.

In [ ]:
from quantem.gpu import DirectPtycho, io
import quantem.widget as qw
from myptycho import Calibration, Dataset, Object, Probe, Propagator, IterativePtycho, show

qw.profile()
data = io.load("scan.qem")
data

In [ ]:
# Set this to the measured rotation if it is missing or must override the file.
# Omit the keyword to reuse a rotation saved in the acquisition metadata.
ptycho = DirectPtycho(data, scan_detector_rotation_deg=0.0)
viewer = qw.ShowPtycho(ptycho, local_tilt=True, show_fft=True, save_dir="ssb-results")
viewer

## Start the selected regions

Finish fitting all selected rectangles first. Each run takes an independent snapshot of its current aberrations, scan-frame tilt, depth spread, rotation and bounds. Manual adjustments are included; another fit is not launched. Moving a rectangle invalidates its fit and the handoff raises until it is refitted.

The full scan is reconstructed with each regional model, then mapped onto that region's iterative object, retaining the surrounding measured phase for the probe footprint. Only the requested scan region is decoded for iterative training, with every native detector pixel. Ten percent of positions are held out for validation.

SSB depth spread is saved as provenance; **it is not the specimen thickness**. The model below supplies thickness explicitly. Each `.npz` stores the independent calibration and full phase; each `.pt` stores the iterative result, initialization record and joint-fit parameters. An unchanged rerun can reuse its matching saved result.

In [ ]:
# Explicit specimen model: depth spread from SSB is not specimen thickness.
results = {}
for region in viewer.regions:
    calibration = Calibration.from_showptycho(viewer, region=region["id"], data=data)
    calibration.save_state(f"ptycho-results/{region['id']}-calibration.npz")
    dataset = Dataset(calibration.crop(data), device=calibration.device)
    run = IterativePtycho(
        dataset, Object(slices=20, thickness_A=250.0, lr=0.008),
        Probe(modes=1, lr=0.001), calibration=calibration,
        propagator=Propagator(lr=0.000075), batch_size=128,
        record_every=4, record_phase=True,
    )
    results[region["label"]] = run.run(8, path=f"ptycho-results/{region['id']}.pt")

## Compare the SSB initialization and iterative output

Epoch 0 is the SSB phase mapped to the iterative object grid, with its regional mean removed and phase divided among slices. Epoch 8 is the trained projection. The arrays, units, and complete joint-fit record are retained in each saved result. Use `mode="static"` in the same `show` call for a static review.

In [ ]:
show(*results.values(), names=list(results), slices="projected", epochs=(0, 8))